# Summary

Modal Parameter Identification by Classical Time-Domain Methods After Blind Machine Learning Denoising

(1) Develop a theoretical 3-DOF model and calculate its modal parameters;  

(2) Generate the system's impulse response;  

2.1 Calculate modal parameters using the classical Complex Exponential method;  

2.2 Calculate modal parameters using the classical Ibrahim method;  

(3) Add noise to the measured data (unit impulse and responses);  

3.1 Study parameter identification by classical methods as the signal-to-noise ratio decreases;  

(4) Implement blind denoising;  

(5) Study parameter identification by classical methods as the signal-to-noise ratio decreases, comparing noisy signals and machine learning model outputs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, stats
import os
import tensorflow as tf
import scipy.linalg
import seaborn as sns

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from typing import List, Tuple, Dict, Any
import matplotlib.ticker as ticker

In [ ]:
# This command lists the physical GPU devices
gpus = tf.config.list_physical_devices('GPU')

# Prints the list of found GPUs
print(gpus)

# A simple test to check if operations can be performed on GPU
if gpus:
    try:
        # Attempt to allocate memory on the first GPU
        tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[0], True)
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
        print("TensorFlow is using GPU!")
    except RuntimeError as e:
        # Memory may already be in use
        print(e)
else:
    print("TensorFlow did NOT find GPU.")

# (1) Develop a theoretical 3-DOF model and calculate its modal parameters;  

In [ ]:
# --- 1. System Parameters and Matrices ---

# Parameters
m = 3.0       # Mass (kg)
m1 = m
m2 = m
m3 = m

# Mass Matrix [M]
M = np.diag([m1, m2, m3])

# Stiffness Constants
k1 = 100  # m1 - ground
k2 = 125  # m1 - m2
k3 = 150  # m2 - m3
k4 = 200  # m3 - ground

# Stiffness Matrix [K]
K = np.array([
    [k1 + k2, -k2, 0],
    [-k2, k2 + k3, -k3],
    [0, -k3, k3 + k4]
])

# Generic (Non-Proportional) Damping
# Viscous Damping Constants
c1 = 5.0  # m1 - ground
c2 = 3.0  # m1 - m2
c3 = 2.0  # m2 - m3
c4 = 1.0  # m3 - ground

# Damping Matrix [C]
C = np.array([
    [c1 + c2, -c2, 0],
    [-c2, c2 + c3, -c3],
    [0, -c3, c3 + c4]
])

print("--- System Matrices ---")
print("Mass Matrix [M]:\n", M)
print("\nStiffness Matrix [K]:\n", K)
print("\nDamping Matrix [C] (Generic):\n", C)

# --- 2. Damped System Analysis (State Matrix) ---

N = 3 # Degrees of Freedom
I = np.identity(N)

# Mass Matrix Inverse
Minv = np.linalg.inv(M)

# State Matrix [A] Submatrices (2N x 2N)
# State defined as z = {x, v}
# [A] = [ [0]   [I]  ]
#       [ -M^-1*K  -M^-1*C ]

Minv_K = Minv @ K
Minv_C = Minv @ C

A_top = np.hstack([np.zeros((N, N)), I])
A_bottom = np.hstack([-Minv_K, -Minv_C])
A = np.vstack([A_top, A_bottom])

# Compute complex eigenvalues and eigenvectors
# eig returns (w, v) where v[:,i] is the eigenvector for eigenvalue w[i]
vals, vecs = np.linalg.eig(A)

# --- 3. Parameter Extraction and Storage ---

print("\n--- Modal Parameters and Mode Shapes ---")

# Lists to store the requested data
nat_freq_list = []       # Natural frequencies (rad/s)
damping_ratio_list = []  # Damping ratios
mode_shapes_list = []    # Mode shapes (complex vectors)

# Filtering and Sorting:
# 1. Identify indices with positive imaginary part (conjugate pairs)
valid_indices = [i for i, val in enumerate(vals) if val.imag > 1e-9]

# 2. Sort these indices by the magnitude of the imaginary part (frequency)
sorted_indices = sorted(valid_indices, key=lambda i: vals[i].imag)

for i, idx in enumerate(sorted_indices):
    lambda_n = vals[idx]          # Eigenvalue
    eigenvector_full = vecs[:, idx] # Complete eigenvector (2N x 1)
    
    # Scalar parameter extraction
    sigma_n = -lambda_n.real        # Exponential decay
    omega_d = lambda_n.imag         # Damped frequency
    
    omega_n_sq = sigma_n**2 + omega_d**2
    omega_n = np.sqrt(omega_n_sq)   # Natural frequency
    zeta_n = sigma_n / omega_n      # Damping factor
    
    # Physical Mode Shape Extraction
    # In state space, the vector is [Displacement; Velocity]
    # Take only the first half (Displacement)
    physical_mode = eigenvector_full[:N]
    
    # Mode Shape Normalization (Optional but recommended for readability)
    # Normalize so the largest element has magnitude 1
    max_val = physical_mode[np.argmax(np.abs(physical_mode))]
    norm_mode = physical_mode / max_val
    
    # --- Save to lists ---
    nat_freq_list.append(omega_n)
    damping_ratio_list.append(zeta_n)
    mode_shapes_list.append(norm_mode)
    
    # --- Print Results ---
    print(f"\nMode {i+1}:")
    print(f"  Natural Frequency (ωn): {omega_n:.4f} rad/s")
    print(f"  Damping Ratio (ζ): {zeta_n:.4f} ({zeta_n*100:.2f}%)")
    print(f"  Damped Frequency (ωd): {omega_d:.4f} rad/s")
    
    # Display formatted complex vector
    mode_str = ", ".join([f"{val:.2f}" for val in norm_mode])
    print(f"  Mode Shape (Normalized): [{mode_str}]")

print("\n--- Summary of Stored Data ---")
print(f"Natural Frequencies List: {nat_freq_list}")
print(f"Damping Ratios List: {damping_ratio_list}")

# (2) Generate the system's impulse response;

In [ ]:
# State-Space Configuration

N = 3
Minv = np.linalg.inv(M)

# State Matrix [A] (6x6)
A_top = np.hstack([np.zeros((N, N)), np.identity(N)])
A_bottom = np.hstack([-Minv @ K, -Minv @ C])  # A_bottom represents the acceleration equation
A = np.vstack([A_top, A_bottom])

# Global Input Matrix [B] (6x3)
B_top = np.zeros((N, N))
B_bottom = Minv  
B_global = np.vstack([B_top, B_bottom])

# --- MODIFICATION 1: Output Matrix for Complete State Return ---
# Instead of returning only position, we return [Position; Velocity] (Identity 6x6)
# This enables accurate acceleration calculation afterward.
C_sys = np.identity(2 * N) 

# --- MODIFICATION 2: Dimensional Adjustment of D ---
# Since C now has 2*N rows, D must also have matching dimensions.
D_global = np.zeros((2 * N, N))

# Time Definition and Simulation
fs = 1024.0          
t_final = 10.0      
t = np.arange(0, t_final, 1/fs)

# Lists to store responses
position_responses = []
acceleration_responses = []  # New list for accelerations

# Loop to simulate one input at a time
for i in range(N):
    B_temp = B_global[:, i:i+1]
    D_temp = D_global[:, i:i+1]
    
    temp_system = signal.StateSpace(A, B_temp, C_sys, D_temp)
    
    # t_out, z_out returns the complete state: [x1, x2, x3, v1, v2, v3]
    t_out, z_out = signal.impulse(temp_system, T=t)
    
    # 1. Extract Position (first N columns)
    pos = z_out[:, :N]
    position_responses.append(pos)
    
    # 2. Calculate Acceleration
    # From state equation: d(state)/dt = A * state + B * u
    # Acceleration is the derivative of velocity (bottom part of state vector).
    # For impulse response, u=0 for t>0.
    # Therefore, acc = A_bottom * state.T
    
    # z_out shape: (time, 6), A_bottom shape: (3, 6)
    # Result shape: (time, 3)
    acc = z_out @ A_bottom.T
    acceleration_responses.append(acc)

# Stack results
# yout (Position): [Time, Output, Input]
yout = np.stack(position_responses, axis=2)

# aout (Acceleration): [Time, Output, Input]
aout = np.stack(acceleration_responses, axis=2)

tout = t

# --- Plotting (Example with Position and Acceleration side by side) ---

fig, axs = plt.subplots(3, 2, figsize=(14, 12), sharex=True)
mass_labels = ['Mass 1', 'Mass 2', 'Mass 3']
colors = ['blue', 'orange', 'green']

for k in range(3):  # Loop through the 3 experiments (inputs)
    
    # Column 1: Position (yout)
    ax_pos = axs[k, 0]
    ax_pos.set_title(f"Exp {k+1}: Position ($F_{k+1} = \delta(t)$)")
    for j in range(3):  # Loop through the 3 masses (outputs)
        ax_pos.plot(tout, yout[:, j, k], label=f'{mass_labels[j]}', color=colors[j])
    ax_pos.set_ylabel("Displacement [m]")
    ax_pos.grid(True, alpha=0.3)
    if k == 0:
        ax_pos.legend(loc="upper right")

    # Column 2: Acceleration (aout)
    ax_acc = axs[k, 1]
    ax_acc.set_title(f"Exp {k+1}: Acceleration ($F_{k+1} = \delta(t)$)")
    for j in range(3):  # Loop through the 3 masses (outputs)
        ax_acc.plot(tout, aout[:, j, k], label=f'{mass_labels[j]}', color=colors[j], linestyle='--')
    ax_acc.set_ylabel("Acceleration [m/s²]")
    ax_acc.grid(True, alpha=0.3)

axs[2, 0].set_xlabel("Time [s]")
axs[2, 1].set_xlabel("Time [s]")

plt.tight_layout()
plt.show()

In [ ]:
np.shape(yout)

In [ ]:
S = np.concatenate((yout, aout), axis=1)

## 2.1 Calculate modal parameters using the classical Complex Exponential method;

In [ ]:
def least_squares_complex_exponential_method(y_signals, fs, order):
    """
    Estimates system poles using the Least Squares Complex Exponential Method (LSCEM).
    Utilizes multiple output channels simultaneously to estimate a global set of poles.
    
    Args:
        y_signals: Array (n_samples, n_channels). If 1D, converted to (n, 1).
        fs: Sampling frequency.
        order: Model order.
        
    Returns:
        poles: Complex poles s (rad/s) of the system.
    """
    # Ensure 2D array (Samples × Channels)
    if y_signals.ndim == 1:
        y_signals = y_signals[:, np.newaxis]
        
    n_samples, n_channels = y_signals.shape
    L = n_samples - order - 1
    
    # Lists to accumulate matrices for each channel
    T_stack = []
    y_target_stack = []
    
    for ch in range(n_channels):
        y_ch = y_signals[:, ch]
        
        # Regression matrix for current channel
        T_ch = np.zeros((L, order))
        for i in range(L):        
            segment = y_ch[i : order + i] 
            T_ch[i, :] = segment[::-1]  
            
        y_target_ch = y_ch[order : order + L]
        
        T_stack.append(T_ch)
        y_target_stack.append(y_target_ch)
    
    # Global vertical stacking (Simultaneous Least Squares)
    # T_global * a = -y_target_global
    T_global = np.vstack(T_stack)
    y_target_global = np.concatenate(y_target_stack)
    
    # Least Squares Solution
    a_coeffs, _, _, _ = np.linalg.lstsq(T_global, -y_target_global, rcond=None)
    
    # Form polynomial and compute roots
    poly_coeffs = np.concatenate(([1.0], a_coeffs))
    roots = np.roots(poly_coeffs)
    roots = roots[np.abs(roots) > 1e-12]  # Remove near-zero roots
    
    # Convert to continuous-time poles
    s_poles = np.log(roots.astype(complex)) * fs
    return s_poles

def get_modal_params(poles):
    """ Converts complex poles s to natural frequencies and damping ratios. 
    
    Args:
        poles: Array of complex poles s (rad/s).
        
    Returns:
        List of dictionaries containing modal parameters for each mode.
    """
    # Filter valid poles (positive imaginary part, stable)
    valid_poles = [s for s in poles if s.imag > 0 and s.real < 1e-6]
    valid_poles.sort(key=lambda x: x.imag)  # Sort by frequency
    
    results = []
    for s in valid_poles:
        wn = np.abs(s)  # Natural frequency (rad/s)
        zeta = -s.real / wn  # Damping ratio
        wd = s.imag  # Damped frequency
        
        results.append({
            'pole': s,
            'wn': wn,
            'fn': wn / (2 * np.pi),  # Natural frequency (Hz)
            'zeta': zeta,
            'wd': wd
        })
    return results

def estimate_residues(yout_experiment, t_vec, poles):
    """ Estimates mode shapes (residues) from impulse response data.
    
    Args:
        yout_experiment: Array (n_samples, n_outputs) - impulse responses.
        t_vec: Time vector (n_samples).
        poles: Complex poles s (rad/s).
        
    Returns:
        residues: Array (n_modes, n_outputs) - complex mode shapes.
    """
    num_modes = len(poles)
    
    # Construct Vandermonde matrix from poles
    V = np.exp(np.outer(t_vec, poles))
    
    num_outputs = yout_experiment.shape[1]
    residues = np.zeros((num_modes, num_outputs), dtype=complex)
    
    # Estimate residues for each output channel
    for i in range(num_outputs):
        y_channel = yout_experiment[:, i]
        r_channel, _, _, _ = np.linalg.lstsq(V, y_channel, rcond=None)
        residues[:, i] = r_channel
        
    return residues

In [ ]:
print("Simulation variables found. Initializing Complex Exponential Method:")

decimation_factor = 20  
fs_id = fs / decimation_factor  # From 1024 Hz to ~51.2 Hz (still well above Nyquist for 2 Hz)

# Decimated reference signal
y_ref_id = S[::decimation_factor, 0:3, 0]

print(f"Original sampling rate: {fs} Hz")
print(f"Sampling rate for identification (Prony): {fs_id} Hz")

# Model Order
# With the decimated signal, the theoretical order (6) works much better.
model_order = 6 

print(f"--- Computing Poles (Order {model_order}) ---")

# Pass the reduced signal and reduced sampling frequency
poles_all = least_squares_complex_exponential_method(y_ref_id, fs_id, model_order)

# Parameter extraction
modal_params = get_modal_params(poles_all)

# Mode Shape Calculation
# For residues, we can use the complete ORIGINAL signal (tout and yout),
# since the poles 's' (rad/s) are universal. This ensures high resolution in modal vectors.
residues_matrix = estimate_residues(S[:, 0:3, 0], tout, poles_all)

print("\n--- Complex Exponential Method Results (LSCEM) ---")

for i, param in enumerate(modal_params):
    pole_val = param['pole']
    
    idx_pole = np.argmin(np.abs(poles_all - pole_val))
    mode_shape = residues_matrix[idx_pole, :]
    
    # Normalization
    norm_factor = mode_shape[0]
    if np.abs(norm_factor) < 1e-10:
        norm_factor = 1.0
        
    mode_shape_norm = mode_shape / norm_factor
    
    print(f"\nMODE {i+1}:")
    print(f"  Natural Frequency (wn):     {param['wn']:.4f} rad/s")
    print(f"  Linear Frequency (fn):      {param['fn']:.2f} Hz")
    print(f"  Damping Ratio (zeta):       {param['zeta']:.4f} ({param['zeta']*100:.2f}%)")
    
    vec_str = ", ".join([f"{c.real:.3f}+{c.imag:.3f}j" for c in mode_shape_norm])
    print(f"  Mode Shape (Norm m1):       [{vec_str}]")

In [ ]:
print("\nGround Truth: Theoretical Modal Parameters")
print(f"Natural Frequencies (ωn) [rad/s]: {nat_freq_list}")
print(f"Damping Ratios (ζ): {damping_ratio_list}")

## 2.2 Calculate modal parameters using the classical Ibrahim method;

In [ ]:
# --- Ibrahim Time Domain (ITD) Method Function ---
def ibrahim_time_domain(responses, fs, delay_block=20, model_order=10):
    """
    Applies the Generalized (Oversized) ITD method for enhanced noise robustness.
    
    Args:
        responses: Array (n_samples, n_dof) containing response signals.
        fs: Sampling frequency (Hz).
        delay_block: Delay between stacked blocks. Increase improves conditioning 
                     at high frequencies.
        model_order: Number of stacked blocks. Increase creates noise subspace.
                     System matrix size will be (model_order * n_dof).
    
    Returns:
        poles: Complex poles s (rad/s) of the system.
        mode_shapes: Complex mode shapes (n_modes, n_dof).
    """
    n_samples, n_dof = responses.shape
    dt_step = 1  # Time shift between Phi and Phi_hat matrices
    
    # Number of available columns
    # Need (model_order * delay_block) samples for vertical stacking
    # plus dt_step for the hat matrix
    rows_total = model_order * n_dof
    L = n_samples - (model_order * delay_block) - dt_step
    
    if L <= rows_total:
        raise ValueError(f"Signal too short for ITD with order {model_order} and delay {delay_block}.")

    # Oversized Phi and Phi_hat matrices
    Phi = np.zeros((rows_total, L))
    Phi_hat = np.zeros((rows_total, L))
    
    data_T = responses.T  # Transpose to (n_dof, n_samples)
    
    # Hankel Matrix Construction by blocks
    for i in range(model_order):
        # Starting index for this block
        start_idx = i * delay_block
        
        # Row range in Phi matrix
        row_start = i * n_dof
        row_end = (i + 1) * n_dof
        
        # Matrix filling
        Phi[row_start:row_end, :] = data_T[:, start_idx : start_idx + L]
        Phi_hat[row_start:row_end, :] = data_T[:, start_idx + dt_step : start_idx + L + dt_step]
    
    # Least Squares Solution for Oversized A matrix
    # A * Phi = Phi_hat
    A_matrix, _, _, _ = np.linalg.lstsq(Phi.T, Phi_hat.T, rcond=None)
    A_matrix = A_matrix.T
    
    # Eigenvalue decomposition
    z_vals, eigenvectors = np.linalg.eig(A_matrix)
    
    # Convert to continuous-time poles
    poles = np.log(z_vals.astype(complex)) * fs
    
    # Mode shape extraction: Take only first n_dof rows of eigenvectors
    # (corresponding to the no-delay block t=0)
    full_modes = eigenvectors[:n_dof, :]
    mode_shapes = full_modes.T
    
    return poles, mode_shapes

# --- Auxiliary Function for Result Processing and Filtering ---
def process_modal_results(poles, mode_shapes, fs):
    """
    Filters physical poles, sorts data, and organizes modal parameters.
    
    Args:
        poles: Array of complex poles s (rad/s).
        mode_shapes: Array of complex mode shapes (n_modes, n_dof).
        fs: Sampling frequency (Hz) for Nyquist limit.
    
    Returns:
        List of dictionaries containing filtered and sorted modal parameters.
    """
    processed_modes = []
    
    for i, pole in enumerate(poles):
        # Initial filtering criteria:
        # 1. Positive frequency (ignore negative conjugates for final list)
        # 2. Non-zero frequency below Nyquist limit
        
        if pole.imag > 1e-1 and pole.imag < fs/2: 
            wn = np.abs(pole)  # Natural frequency (rad/s)
            zeta = -pole.real / wn  # Damping ratio
            
            # --- REQUESTED FILTERING ---
            # Remove modes with zeta >= 1 (overdamped / non-oscillatory)
            if zeta < 1.0:
                wd = pole.imag  # Damped frequency
                fn = wn / (2 * np.pi)  # Natural frequency in Hz
                
                # Get corresponding mode shape
                mode = mode_shapes[i, :]
                
                processed_modes.append({
                    'pole': pole,
                    'wn': wn,
                    'fn': fn,
                    'zeta': zeta,
                    'mode_shape': mode
                })
            
    # Sort by natural frequency
    processed_modes.sort(key=lambda x: x['wn'])
    
    return processed_modes

In [ ]:
print("Starting Modal Analysis via ITD (Ibrahim Time Domain)")

# 1. Data Selection
# The ITD method is SIMO/MIMO. We can use all sensor responses
# for a specific excitation input.
# Using Experiment 1: Impulse on m1 (S[:, 0:3, 0])
responses_exp1 = S[:, 0:3, 0]  # Shape: (Time, 3 Masses)

# 2. Time Delay Block Definition
# Should be chosen such that signals are linearly independent.
# 10 samples @ 1024Hz = 0.1s.
delay_samples = 10 

# 3. Algorithm Execution
poles_itd, modes_itd = ibrahim_time_domain(responses_exp1, fs, delay_samples)

# 4. Result Processing
results = process_modal_results(poles_itd, modes_itd, fs)

print("\n--- Ibrahim Time Domain Method Results ---")

# Duplicate Filtering (Close pairs may appear due to numerical noise or ill-conditioned matrix)
# In this ideal case, should return exactly 3 modes.
unique_results = []
last_fn = -1
for res in results:
    if (abs(res['fn'] - last_fn) > 0.1) & (res['zeta'] < 0.98):  # 0.1 Hz tolerance for distinct modes
        unique_results.append(res)
        last_fn = res['fn']

for i, res in enumerate(unique_results):
    print(f"\nMODE {i+1}:")
    print(f"  Natural Frequency (wn):     {res['wn']:.4f} rad/s")
    print(f"  Linear Frequency (fn):      {res['fn']:.2f} Hz")
    print(f"  Damping Ratio (zeta):       {res['zeta']:.4f} ({res['zeta']*100:.2f}%)")
    
    # Mode Shape Normalization (by first mass m1)
    mode = res['mode_shape']
    norm_val = mode[0] if np.abs(mode[0]) > 1e-6 else 1.0
    mode_norm = mode / norm_val
    
    vec_str = ", ".join([f"{c.real:.3f}+{c.imag:.3f}j" for c in mode_norm])
    print(f"  Mode Shape (Norm m1):       [{vec_str}]")

In [ ]:
print("\nGround Truth: Theoretical Modal Parameters")
print(f"Natural Frequencies (ωn) [rad/s]: {nat_freq_list}")
print(f"Damping Ratios (ζ): {damping_ratio_list}")

# (3) Add noise to the measured data (unit impulse and responses);

In [ ]:
def add_noise_to_yout(yout_data, snr_db):
    """
    Adiciona ruído Gaussiano aos sinais de saída (yout) baseado na relação sinal-ruído (SNR).
    
    Args:
        yout_data: Array (n_samples, n_outputs, n_inputs) com os dados da simulação.
        snr_db: Relação Sinal-Ruído em decibéis (dB).
    
    Returns:
        yout_noisy: Cópia do array original com ruído adicionado.
    """
    # Cria uma cópia para não alterar o original
    yout_noisy = np.copy(yout_data)
    n_inputs = yout_data.shape[2]
    n_outputs = yout_data.shape[1]
    n_samples = yout_data.shape[0]
    
    # O ruído é calculado para cada canal individualmente para manter a SNR consistente
    for i in range(n_inputs): # Para cada experimento (entrada)
        for j in range(n_outputs): # Para cada sensor (saída)
            signal_vec = yout_data[:, j, i]
            
            # Calcula o desvio padrão do sinal (representando a amplitude RMS AC)
            sig_std = np.std(signal_vec)
            
            # Evita divisão por zero ou ruído em canais mortos
            if sig_std < 1e-10:
                continue
                
            # Calcula o desvio padrão do ruído necessário a partir da SNR
            # Fórmula: SNR_dB = 20 * log10(sig_std / noise_std)
            # Logo: noise_std = sig_std / (10^(SNR_dB/20))
            noise_std = sig_std / (10**(snr_db/20.0))
            
            # Gera ruído branco gaussiano
            noise = np.random.normal(loc=0.0, scale=noise_std, size=n_samples)
            
            # Adiciona ao sinal
            yout_noisy[:, j, i] += noise
            
    return yout_noisy

In [ ]:
def add_noise_to_yout(yout_data, snr_db):
    """
    Adds Gaussian noise to output signals (yout) based on the signal-to-noise ratio (SNR).
    
    Args:
        yout_data: Array (n_samples, n_outputs, n_inputs) containing simulation data.
        snr_db: Signal-to-Noise Ratio in decibels (dB).
    
    Returns:
        yout_noisy: Copy of the original array with added noise.
    """
    # Create a copy to avoid modifying the original data
    yout_noisy = np.copy(yout_data)
    n_inputs = yout_data.shape[2]
    n_outputs = yout_data.shape[1]
    n_samples = yout_data.shape[0]
    
    # Noise is calculated for each channel individually to maintain consistent SNR
    for i in range(n_inputs):  # For each experiment (input)
        for j in range(n_outputs):  # For each sensor (output)
            signal_vec = yout_data[:, j, i]
            
            # Calculate signal standard deviation (representing AC RMS amplitude)
            sig_std = np.std(signal_vec)
            
            # Avoid division by zero or noise in dead channels
            if sig_std < 1e-10:
                continue
                
            # Calculate required noise standard deviation from SNR
            # Formula: SNR_dB = 20 * log10(sig_std / noise_std)
            # Therefore: noise_std = sig_std / (10^(SNR_dB/20))
            noise_std = sig_std / (10**(snr_db / 20.0))
            
            # Generate white Gaussian noise
            noise = np.random.normal(loc=0.0, scale=noise_std, size=n_samples)
            
            # Add to signal
            yout_noisy[:, j, i] += noise
            
    return yout_noisy

In [ ]:
## Example Results with SNR:
snr_db = 20

# Calculate theoretical eigenvalues and modal parameters
eigenvalues_real, _ = np.linalg.eig(A)
params_real = get_modal_params(eigenvalues_real)

lista_wn_real = []
lista_zeta_real = []
filtered_real_modes = []

last_wn = -1
for p in params_real:
    if p['wn'] > 0.1 and abs(p['wn'] - last_wn) > 0.1:  # Simple uniqueness filter
        lista_wn_real.append(p['wn'])
        lista_zeta_real.append(p['zeta'])
        filtered_real_modes.append(p)
        last_wn = p['wn']

print("Theoretical Parameters (Reference):")
print(f"  Natural Frequencies (wn): {np.round(lista_wn_real, 4)} rad/s")
print(f"  Damping Ratios (ζ): {np.round(lista_zeta_real, 4)}")

# 3.2. Test Loop Configuration
snr = snr_db
print(f"\nProcessing SNR = {snr} dB...")

# Add noise to signals
yout_noisy = add_noise_to_yout(yout, snr)

# --- METHOD 1: COMPLEX EXPONENTIAL METHOD (LSCEM) ---
# Requires downsampling for stability
decimation = 20
fs_cem = fs / decimation

y_signals_lscem = yout_noisy[::decimation, :, 0]
results_table = []

poles_all = least_squares_complex_exponential_method(y_signals_lscem, fs_cem, model_order)
modal_params = get_modal_params(poles_all)

print("\n--- Complex Exponential Method Results (LSCEM) ---")

for i, param in enumerate(modal_params):
    pole_val = param['pole']
    
    idx_pole = np.argmin(np.abs(poles_all - pole_val))
    mode_shape = residues_matrix[idx_pole, :]
    
    # Normalization
    norm_factor = mode_shape[0]
    if np.abs(norm_factor) < 1e-10:
        norm_factor = 1.0
        
    mode_shape_norm = mode_shape / norm_factor
    
    print(f"\nMODE {i+1}:")
    print(f"  Natural Frequency (wn):     {param['wn']:.4f} rad/s")
    print(f"  Linear Frequency (fn):      {param['fn']:.2f} Hz")
    print(f"  Damping Ratio (zeta):       {param['zeta']:.4f} ({param['zeta']*100:.2f}%)")
    
    vec_str = ", ".join([f"{c.real:.3f}+{c.imag:.3f}j" for c in mode_shape_norm])
    print(f"  Mode Shape (Norm m1):       [{vec_str}]")

# --- METHOD 2: IBRAHIM TIME DOMAIN (ITD) ---
# Uses complete signal (all outputs from experiment 1)
delay_samples = 20 
responses_itd = yout_noisy[:, :, 0]

# Execute algorithm
poles_itd, modes_itd = ibrahim_time_domain(responses_itd, fs, delay_samples)

# Process results
results = process_modal_results(poles_itd, modes_itd, fs)

print("\n--- Ibrahim Time Domain Method Results ---")

# Duplicate filtering (Close pairs may appear due to numerical noise or ill-conditioned matrix)
# In ideal conditions, should return exactly 3 modes.
unique_results = []
last_fn = -1
for res in results:
    if (abs(res['fn'] - last_fn) > 0.1) & (res['zeta'] < 0.98):  # 0.1 Hz tolerance for distinct modes
        unique_results.append(res)
        last_fn = res['fn']

for i, res in enumerate(unique_results):
    print(f"\nMODE {i+1}:")
    print(f"  Natural Frequency (wn):     {res['wn']:.4f} rad/s")
    print(f"  Linear Frequency (fn):      {res['fn']:.2f} Hz")
    print(f"  Damping Ratio (zeta):       {res['zeta']:.4f} ({res['zeta']*100:.2f}%)")
    
    # Mode shape normalization (by first mass m1)
    mode = res['mode_shape']
    norm_val = mode[0] if np.abs(mode[0]) > 1e-6 else 1.0
    mode_norm = mode / norm_val
    
    vec_str = ", ".join([f"{c.real:.3f}+{c.imag:.3f}j" for c in mode_norm])
    print(f"  Mode Shape (Norm m1):       [{vec_str}]")

## 3.1 Study parameter identification by classical methods as the signal-to-noise ratio decreases;

# (4) Implement blind denoising;

In [ ]:
def create_lagged_dataframe(
    models_data: Dict[Tuple, Dict[str, Any]], 
    num_points: int
) -> pd.DataFrame:
    """
    Converts a dictionary of experiments into a single Pandas DataFrame 
    with lagged (past) and leaded (future) features.

    Each experiment in the input dictionary is processed independently. For each
    signal (e.g., 'x'), it creates new columns for past and future values.
    For example, if num_points = 2, it will generate:
    'x_m_2', 'x_m_1', 'x', 'x_p_1', 'x_p_2'

    Boundary Conditions:
    For points at the beginning or end of a sequence where past or future
    data is not available, the value of the nearest boundary point (the
    first or last data point in that sequence) is used for padding.

    Args:
        models_data: The input dictionary of dictionaries.
            Example: { ('Hunt_Crossley', 0.1, 0.76): {'x': [..], 'v': [..], ...} }
        num_points: The number of past and future points to include as
            features (a non-negative integer).

    Returns:
        A single Pandas DataFrame containing the time-expanded features
        from all experiments. Includes 'experiment_key' and 
        'experiment_name' columns to identify the source of each row.
    """
    
    # A list to store the individual DataFrames for each experiment
    all_experiment_dfs = []

    # Iterate through each experiment in the top-level dictionary
    for experiment_key, signals_dict in models_data.items():
        try:
            # Convert the inner dictionary of signals into a DataFrame
            # This aligns all signals by their index (time step)
            experiment_df = pd.DataFrame(signals_dict)
            
            # Skip empty experiments
            if experiment_df.empty:
                print(f"Skipping empty experiment: {experiment_key}")
                continue

            # This will be the new DataFrame for this single experiment
            # with all the lagged/leaded columns
            lagged_df_single_experiment = pd.DataFrame()

            # Get the names of all signals to process (e.g., 'x', 'v', 'f_kN')
            signal_columns = experiment_df.columns

            for col_name in signal_columns:
                original_series = experiment_df[col_name]
                
                # Get the first and last values for boundary padding
                first_val = original_series.iloc[0]
                last_val = original_series.iloc[-1]

                # 1. Add Past (Minus) columns
                # e.g., for num_points=3, this loops i=3, 2, 1
                for i in range(num_points, 0, -1):
                    new_col_name = f"{col_name}_m_{i}"
                    # shift(i) moves data down, creating NaNs at the top.
                    # fill_value=first_val handles the start boundary condition.
                    lagged_df_single_experiment[new_col_name] = original_series.shift(
                        periods=i, fill_value=first_val
                    )

                # 2. Add Current value column
                lagged_df_single_experiment[col_name] = original_series

                # 3. Add Future (Plus) columns
                # e.g., for num_points=3, this loops i=1, 2, 3
                for i in range(1, num_points + 1):
                    new_col_name = f"{col_name}_p_{i}"
                    # shift(-i) moves data up, creating NaNs at the bottom.
                    # fill_value=last_val handles the end boundary condition.
                    lagged_df_single_experiment[new_col_name] = original_series.shift(
                        periods=-i, fill_value=last_val
                    )

            # Add identifier columns to know which experiment this data came from
            lagged_df_single_experiment['experiment_key'] = str(experiment_key)
            if len(experiment_key) > 0:
                lagged_df_single_experiment['experiment_name'] = experiment_key[0]
            
            # Add the fully processed DataFrame for this experiment to our list
            all_experiment_dfs.append(lagged_df_single_experiment)

        except Exception as e:
            print(f"Warning: Could not process experiment {experiment_key}. Error: {e}")
            continue

    # If no data was processed, return an empty DataFrame
    if not all_experiment_dfs:
        return pd.DataFrame()

    # Concatenate all individual experiment DataFrames into one large one
    final_df = pd.concat(all_experiment_dfs, ignore_index=True)
    
    # Optional: Re-order columns to put identifiers first
    id_cols = [col for col in final_df.columns if 'experiment_' in col]
    data_cols = [col for col in final_df.columns if col not in id_cols]
    final_df = final_df[id_cols + data_cols]

    return final_df

def convert_yout_to_dict(yout_data):
    """
    Converts 3D 'yout' array (Time, Outputs, Inputs) to dictionary format
    expected by the create_lagged_dataframe function.
    
    Expected format: { ('Experiment_Name', param1...): {'signal1': [...], 'signal2': [...]} }
    
    Args:
        yout_data: 3D array of shape (n_samples, n_outputs, n_inputs) containing 
                   time-domain responses.
    
    Returns:
        models_data: Dictionary with experiment keys and signal dictionaries.
    """
    models_data = {}
    n_inputs = yout_data.shape[2]   # Number of experiments (inputs)
    n_outputs = yout_data.shape[1]  # Number of signals per experiment (outputs/masses)
    
    # Output signal names (e.g., x1, x2, x3 for displacements)
    signal_names = [f"x{i+1}" for i in range(n_outputs)]
    
    for i in range(n_inputs):
        # Unique key for the experiment
        # Example: ('Impulse_Test', 'Input_Mass_1')
        exp_key = ('Impulse_Test', f'Input_Mass_{i+1}')
        
        # Inner dictionary for signals in this experiment
        signals_dict = {}
        for j in range(n_outputs):
            # Extract time series for each signal
            signals_dict[signal_names[j]] = yout_data[:, j, i]
            
        models_data[exp_key] = signals_dict
        
    return models_data



In [ ]:
print("\n=== Generating Lagged DataFrame with Noise ===")

# 1. Define noise level for DataFrame
SNR_DATAFRAME = 20  # dB

# 2. Generate noisy data
S_for_df = add_noise_to_yout(S, SNR_DATAFRAME)

# 3. Convert to dictionary format
dict_data = convert_yout_to_dict(S_for_df)

# 4. Create DataFrame with Lags (Time window)
# num_points=2 generates: x_m_2, x_m_1, x, x_p_1, x_p_2
lagged_df = create_lagged_dataframe(dict_data, num_points=2)

print("DataFrame successfully generated!")
print(f"Dimensions: {lagged_df.shape}")
print(f"Generated columns (Sample): {list(lagged_df.columns[:10])} ...")
print("\nFirst 5 rows:")
print(lagged_df.head())

In [ ]:
lagged_df.columns

In [ ]:
lagged_df[['experiment_key', 'experiment_name']].value_counts()

In [ ]:
from tensorflow.keras.callbacks import Callback

class StopAfterOverfitting(Callback):
    """
    Custom callback that stops training only after genuine overfitting is detected.
    
    Overfitting is defined by two concurrent conditions:
    1. The validation error (monitored metric) has not improved for 'patience' epochs.
    2. The current validation error is strictly greater than the current training error.
    
    This dual criterion prevents premature stopping when the model is still learning
    and ensures termination only when generalization performance deteriorates.
    """
    
    def __init__(self, monitor='val_loss', patience=100, verbose=1, restore_best_weights=True):
        super(StopAfterOverfitting, self).__init__()
        self.monitor = monitor
        self.patience = patience
        self.verbose = verbose
        self.restore_best_weights = restore_best_weights
        
        # Initialize control variables
        self.wait = 0
        self.stopped_epoch = 0
        self.best_weights = None
        self.best_loss = np.Inf

    def on_train_begin(self, logs=None):
        # Reset variables at training start
        self.wait = 0
        self.stopped_epoch = 0
        self.best_loss = np.Inf
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        current_val_loss = logs.get(self.monitor)
        current_train_loss = logs.get('loss')

        if current_val_loss is None:
            return

        # Check if validation error improved
        if current_val_loss < self.best_loss:
            self.best_loss = current_val_loss
            self.wait = 0
            if self.restore_best_weights:
                self.best_weights = self.model.get_weights()
        else:
            self.wait += 1

        # Dual stopping condition:
        # 1. Patience threshold exceeded
        # 2. Validation loss is strictly greater than training loss
        if self.wait >= self.patience and current_val_loss > current_train_loss:
            self.stopped_epoch = epoch
            self.model.stop_training = True
            if self.restore_best_weights and self.best_weights is not None:
                if self.verbose > 0:
                    print("   > Overfitting detected (Val Loss > Train Loss). Restoring model weights from the best epoch.")
                self.model.set_weights(self.best_weights)

    def on_train_end(self, logs=None):
        if self.stopped_epoch > 0 and self.verbose > 0:
            print(f"Training terminated early at epoch {self.stopped_epoch + 1} due to the custom overfitting condition.")

In [ ]:
def _build_model_from_config(config: Dict[str, Any]) -> Model:
    """
    Internal helper function to build a Keras model from a configuration dictionary.

    This function creates an encoder and a mirrored decoder structure.
    The layer sizes are defined as integer multiples of the input feature dimension.
    All hidden layers use 'tanh' activation.
    """

    latent_dim = config['latent_dim']
    num_features_in = config['num_features_in']
    num_features_out = config['num_features_out']

    input_layer = Input(shape=(num_features_in,))
    
    # --- Encoder ---
    # The chain of layers 'x' will lead to the latent space
    x = input_layer 
    
    x = Dense(int(num_features_in * 4+6), activation='tanh')(x)
    x = Dense(int(num_features_in * 2+6), activation='tanh')(x)
    x = Dense(int(num_features_in * 2+6), activation='tanh')(x)
    x = Dense(np.min([int(num_features_in * 0.5),2*latent_dim+6]), activation='tanh')(x)
    
    latent_space = Dense(latent_dim, activation='tanh')(x)
    x = latent_space

    x = Dense(np.min([int(num_features_in * 0.5),2*latent_dim+6]), activation='tanh')(x)
    x = Dense(int(num_features_in * 1+6), activation='tanh')(x)
    x = Dense(int(num_features_in * 2+6), activation='tanh')(x)
    x = Dense(int(num_features_in * 2+6), activation='tanh')(x)
        
    output_layer = Dense(num_features_out, activation='linear')(x)
    
    model = Model(input_layer, output_layer, name='Blind_Denoising_' + f"{latent_dim}D")
    return model

In [ ]:
def _build_model_from_config_autoencoder(config: Dict[str, Any]) -> Model:
    """
    Internal helper function to build a Keras model from a configuration dictionary.

    This function creates an encoder and a mirrored decoder structure.
    The layer sizes are defined as integer multiples of the input feature dimension.
    All hidden layers use 'tanh' activation.
    """

    latent_dim = config['latent_dim']
    num_features_in = config['num_features_in']
    num_features_out = config['num_features_out']

    input_layer = Input(shape=(num_features_in,))
    
    # --- Encoder ---
    # The chain of layers 'x' will lead to the latent space
    x = input_layer 
    
    x = Dense(int(num_features_in * 19), activation='tanh')(x)
    x = Dense(int(num_features_in * 9), activation='tanh')(x)
    x = Dense(int(num_features_in * 9), activation='tanh')(x)
    x = Dense(int(2*latent_dim), activation='tanh')(x)
    
    latent_space = Dense(latent_dim, activation='tanh')(x)
    x = latent_space

    x = Dense(int(2*latent_dim), activation='tanh')(x)
    x = Dense(int(num_features_in * 4), activation='tanh')(x)
    x = Dense(int(num_features_in * 9), activation='tanh')(x)
    x = Dense(int(num_features_in * 9), activation='tanh')(x)
        
    output_layer = Dense(num_features_out, activation='linear')(x)
    
    model = Model(input_layer, output_layer, name='Blind_Denoising_' + f"{latent_dim}D")
    return model

In [ ]:
def train_final_model(
    untrained_model: Model, 
    full_dataframe: pd.DataFrame, 
    input_variables: List[str], 
    output_variables: List[str]
) -> Tuple[Model, pd.DataFrame]:
    """
    Trains a given model architecture on the *full* dataset with
    final hyperparameters (LR=0.001, 5000 epochs, 25% batch size).
    
    A validation split (10%) is created to monitor for overfitting.

    Args:
        untrained_model: The Keras Model (untrained) from the search.
        full_dataframe: The complete DataFrame to train on.
        input_variables: List of column names to use as inputs.
        output_variables: List of column names to use as outputs.

    Returns:
        A tuple containing:
        1. (Model): The fully trained Keras model.
        2. (pd.DataFrame): 'Y_output_model', a DataFrame of the model's
                           predictions on the *entire* input dataset.
    """
    
    print("\n--- Starting Final Training ---")
    print(f"Training model {untrained_model.name} on {len(full_dataframe)} samples.")

    # 1. Final Hyperparameters
    LEARNING_RATE = 0.001
    EPOCHS = 5000
    LOSS_FUNCTION = 'mean_absolute_error'
    
    # 2. Prepare full dataset
    X_data = full_dataframe[input_variables]
    y_data = full_dataframe[output_variables]

    # 3. Scaling (on 100% of data)
    scaler_X = StandardScaler()
    X_scaled = scaler_X.fit_transform(X_data)
    
    scaler_y = StandardScaler()
    y_scaled = scaler_y.fit_transform(y_data)

    # 4. Create Train/Validation split for final training
    # This is necessary for the StopAfterOverfitting callback and the plot
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_scaled, test_size=0.1, random_state=42
    )

    batch_size = 32
    print(f"Final params: Epochs={EPOCHS}, LR={LEARNING_RATE}, BatchSize={batch_size}")
    print(f"Using {len(X_train)} samples for training, {len(X_val)} for validation.")

    # 6. Re-compile the model with the new learning rate
    optimizer = Adam(learning_rate=LEARNING_RATE)
    untrained_model.compile(optimizer=optimizer, loss=LOSS_FUNCTION)
    
    # 7. --- Callbacks ---
    # Using the patience of 100 from your provided snippet
    stop_callback = StopAfterOverfitting(
        monitor='val_loss', 
        patience=100, 
        restore_best_weights=True, 
        verbose=1
    )
    
    # 8. Fit the model on the training data, validating on the validation data
    history = untrained_model.fit(
        X_train, 
        y_train,
        validation_data=(X_val, y_val), # Added validation data
        epochs=EPOCHS,
        batch_size=batch_size,
        callbacks=[stop_callback],
        verbose=0
    )
    
    print("--- Final Training Complete ---")

    # --- 9. Generate and Save Plot  ---
    plt.figure(figsize=(10, 6))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    
    # Find and mark the best epoch
    best_epoch_index = np.argmin(history.history['val_loss'])
    plt.axvline(x=best_epoch_index, color='r', linestyle='--', 
                label=f'Best Epoch ({best_epoch_index + 1})')
    
    plt.title(f'Final Model Training History ({untrained_model.name})')
    plt.xlabel('Epoch')
    plt.ylabel(LOSS_FUNCTION)
    plt.yscale('log')
    plt.legend()
    plt.grid(True)
    
    plot_filename = "final_model_training_history.png"
    plt.savefig(plot_filename)
    print(f"Final training history plot saved to {plot_filename}")

    # --- 10. Generate Y_output_model  ---
    print("Generating predictions for the full dataset...")
    # Predict on the *entire* scaled dataset
    y_pred_scaled = untrained_model.predict(X_scaled)
    
    # Inverse transform (un-scale) the predictions
    y_pred_unscaled = scaler_y.inverse_transform(y_pred_scaled)
    
    # Create the final DataFrame
    Y_output_model = pd.DataFrame(
        y_pred_unscaled, 
        columns=output_variables
    )

    # 11. Return the trained model and the output DataFrame
    # The 'untrained_model' variable now holds the trained weights
    return untrained_model, Y_output_model

In [ ]:
# Standard Autoencoder
input_cols = ['x1','x2','x3','x4','x5','x6']

output_cols = ['x1','x2','x3','x4','x5','x6']


all_unique_cols = list(set(input_cols + output_cols))

config = {  'latent_dim' : 4,
            'num_features_in' :6, 
            'num_features_out': 6}

model = _build_model_from_config_autoencoder(config)

model.summary()

print(f"Treinando modelo")
trained_modtael, Y_output_model = train_final_model(
    untrained_model=model,
    full_dataframe=lagged_df,
    input_variables=input_cols,
    output_variables=output_cols
)

# --- LÓGICA DE RESHAPE ---
# O DataFrame Y_output_model contém todos os experimentos empilhados verticalmente.
n_samples_per_exp = len(t)
n_experiments = 3 # Três massas sendo excitadas individualmente
n_outputs = 6     # Três respostas (x1, x2, x3)

# Extrai valores como numpy array
# Shape atual: (Total_Samples, n_outputs) -> (46080, 3)
y_flat = Y_output_model.values

# 1. Quebra em blocos por experimento
# Novo Shape: (n_experiments, n_samples, n_outputs) -> (3, 15360, 3)
y_reshaped_temp = y_flat.reshape((n_experiments, n_samples_per_exp, n_outputs))

# 2. Transpõe para o formato padrão 'yout' (Tempo, Outputs, Inputs)
# Formato yout: (15360, 3, 3)
S_denoised = y_reshaped_temp.transpose(1, 2, 0)
yout_denoised = S_denoised[:,0:3,:]

# --- CÁLCULO DE SNR DO OUTPUT ---
# O SNR de saída é calculado comparando o sinal denoisado (predito) com o sinal limpo original (yout)
# O "ruído" residual é a diferença entre o que a rede entregou e o sinal perfeito.

# Cálculo Global (considerando todos os pontos de todos os sinais)
residual_noise = yout_denoised - yout
std_signal_clean = np.std(yout)
std_residual = np.std(residual_noise)

# Fórmula: SNR = 20 * log10(Std_Sinal / Std_Residual)
snr_output_db = 20 * np.log10(std_signal_clean / std_residual)

print(f"\n--- Análise de Desempenho do Denoising ---")
print(f"SNR de Entrada (Definido): {SNR_DATAFRAME:.2f} dB")
print(f"SNR de Saída (Calculado):  {snr_output_db:.2f} dB")
print(f"Ganho de SNR:              {snr_output_db - SNR_DATAFRAME:.2f} dB")

# --- PLOTAGEM ---
fig, axs = plt.subplots(3, 1, figsize=(10, 12), sharex=True)
massas_labels = ['Massa 1', 'Massa 2', 'Massa 3']
cores = ['blue', 'orange', 'green']

# Plotagem dos 3 Experimentos (Entradas)
for input_idx in range(3):
    ax = axs[input_idx]
    ax.set_title(f"Resultado Denoising - Experimento {input_idx+1}: Impulso em $m_{input_idx+1}$")
    
    for output_idx in range(3):
        # Sinal Ruidoso (Original do DF)
        # ax.plot(tout, yout_for_df[:, output_idx, input_idx], color='lightgray', alpha=0.5, linewidth=1)
        
        # Sinal Predito pela Rede (Denoised)
        ax.plot(tout, yout_denoised[:, output_idx, input_idx], 
                label=f'Pred. {massas_labels[output_idx]}', 
                color=cores[output_idx], 
                linestyle='--')
        
    ax.set_ylabel("Deslocamento [m]")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")

axs[2].set_xlabel("Tempo [s]")
plt.tight_layout()
plt.show()

print("Processo concluído.")

In [ ]:
# 1. Define the input and output columns as specified
input_cols = [
    'x1_m_2', 'x1_m_1', 'x1_p_1', 'x1_p_2',
    'x2_m_2', 'x2_m_1', 'x2_p_1', 'x2_p_2',
    'x3_m_2', 'x3_m_1', 'x3_p_1', 'x3_p_2',
    'x4_m_2', 'x4_m_1', 'x4_p_1', 'x4_p_2',
    'x5_m_2', 'x5_m_1', 'x5_p_1', 'x5_p_2',
    'x6_m_2', 'x6_m_1', 'x6_p_1', 'x6_p_2'
    ]

output_cols = ['x1','x2','x3','x4','x5','x6']


all_unique_cols = list(set(input_cols + output_cols))

config = {  'latent_dim' : 4,
            'num_features_in' :24, 
            'num_features_out': 6}

model = _build_model_from_config(config)

model.summary()

print(f"Treinando modelo")
trained_modtael, Y_output_model = train_final_model(
    untrained_model=model,
    full_dataframe=lagged_df,
    input_variables=input_cols,
    output_variables=output_cols
)

In [ ]:
Y_output_model.shape

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- DATA RESHAPING LOGIC ---
# The 'Y_output_model' DataFrame contains all experiments stacked vertically.
# We need to reconstruct the standard state-space 3D tensor: (Time, Outputs, Inputs).

n_samples_per_exp = len(t)
n_experiments = 3  # Number of independent excitation experiments (Inputs)
n_outputs = 6      # Number of sensor responses (Outputs: x1, x2, x3, etc.)

# Extract raw values as a numpy array
# Current Shape: (Total_Samples, n_outputs) -> e.g., (46080, 6)
y_flat = Y_output_model.values

# 1. Reshape into blocks per experiment
# Intermediate Shape: (n_experiments, n_samples, n_outputs) -> (3, 15360, 6)
y_reshaped_temp = y_flat.reshape((n_experiments, n_samples_per_exp, n_outputs))

# 2. Transpose to the standard 'yout' format used in Control Systems: (Time, Outputs, Inputs)
# Final Shape: (15360, 6, 3)
S_denoised = y_reshaped_temp.transpose(1, 2, 0)

# Slice to keep only displacement states (first 3 outputs) if necessary
yout_denoised = S_denoised[:, 0:3, :]

# --- OUTPUT SNR CALCULATION ---
# The Output SNR quantifies the reconstruction quality.
# The "Residual Noise" is defined as the difference between the Denoised Prediction 
# and the clean Ground Truth (yout).

# Global Calculation (aggregating all time steps and channels)
residual_noise = yout_denoised - yout  # Error signal
std_signal_clean = np.std(yout)
std_residual = np.std(residual_noise)

# Formula: SNR_dB = 20 * log10(Signal_RMS / Noise_RMS)
snr_output_db = 20 * np.log10(std_signal_clean / std_residual)

print(f"\n--- Denoising Performance Analysis ---")
print(f"Input SNR (Target):      {SNR_DATAFRAME:.2f} dB")
print(f"Output SNR (Calculated): {snr_output_db:.2f} dB")
print(f"SNR Gain:                {snr_output_db - SNR_DATAFRAME:.2f} dB")

# --- VISUALIZATION ---
fig, axs = plt.subplots(3, 1, figsize=(10, 12), sharex=True)
mass_labels = ['Mass 1', 'Mass 2', 'Mass 3']
colors = ['blue', 'orange', 'green']

# Plot results for each Experiment (Input Excitation)
for input_idx in range(3):
    ax = axs[input_idx]
    ax.set_title(f"Denoising Result - Experiment {input_idx+1}: Impulse on $m_{input_idx+1}$")
    
    for output_idx in range(3):
        # Optional: Plot the original Noisy Input for comparison
        # ax.plot(tout, yout_for_df[:, output_idx, input_idx], color='lightgray', alpha=0.5, linewidth=1)
        
        # Plot the Denoised Model Prediction
        ax.plot(tout, yout_denoised[:, output_idx, input_idx], 
                label=f'Pred. {mass_labels[output_idx]}', 
                color=colors[output_idx], 
                linestyle='--')
        
    ax.set_ylabel("Displacement [m]")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")

axs[2].set_xlabel("Time [s]")
plt.tight_layout()
plt.show()

print("Process completed successfully.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Scientific Plotting Style Configuration ---
# Standardizes the visual output for academic publications (IEEE/Nature style)
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'], 
    'mathtext.fontset': 'stix',          # Use STIX fonts for LaTeX-like math rendering
    'axes.linewidth': 1.2,               # Slightly reduced to match the smaller figure size
    'xtick.direction': 'in',             # Ticks pointing inwards (standard in physics/engineering)
    'ytick.direction': 'in',           
    'xtick.major.size': 5,             
    'ytick.major.size': 5,
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,
    'xtick.top': True,                   # Ticks on top/right spines for framing
    'ytick.right': True,               
    'lines.linewidth': 2.0,              # Thicker lines for readability on smaller figures
    'legend.frameon': True,            
    'legend.framealpha': 0.9,          
    'legend.edgecolor': 'black',       
    'legend.fancybox': False,            # Square corners for the legend box
    'figure.dpi': 150                    # High DPI for clear preview
})

# Font size configuration adjusted for the specific figure aspect ratio (8x3.2 inches)
FONT_CONFIG = {
    'axis_label_size': 14, # Reduced from 20 to maintain proportion
    'tick_label_size': 12, # Reduced from 16
    'legend_size': 11,     # Reduced from 16
}

# Professional Color Palette (High Contrast & Colorblind-friendly)
colors = ['#0072B2', '#D55E00', '#009E73'] 

def format_scientific_axes(ax):
    """
    Applies standard formatting to Matplotlib axes for scientific figures:
    - Enables minor ticks
    - Sets grid styles (Major: dashed, Minor: dotted)
    - Standardizes tick label sizes and padding
    """
    ax.minorticks_on() 
    ax.grid(which='major', linestyle='--', linewidth=0.7, alpha=0.5, color='gray')
    ax.grid(which='minor', linestyle=':', linewidth=0.4, alpha=0.3, color='gray')
    ax.tick_params(axis='both', which='major', labelsize=FONT_CONFIG['tick_label_size'], pad=6)
    
    # Ensure all spines (borders) have consistent thickness
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)

# Figure Dimensions Strategy:
# Width: 8 inches (spans full width of standard A4/Letter paper with margins)
# Height: 3.2 inches per plot.
# Stacked vertically (3 plots), this equals ~9.6 inches, fitting comfortably on one A4 page.
FIG_DIMENSIONS = (8, 3.2)

# -------------------------------------------------------------------------
# PLOT 1: GROUND TRUTH (Clean Data)
# -------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=FIG_DIMENSIONS)

mass_labels = ['Mass 1', 'Mass 2', 'Mass 3']

for i in range(3):
    ax.plot(tout, yout[:, i, 0], 
            label=f'{mass_labels[i]}', 
            color=colors[i],
            linestyle='-', 
            linewidth=plt.rcParams['lines.linewidth'])

ax.set_ylabel("Displacement [m]", fontsize=FONT_CONFIG['axis_label_size'])
ax.set_xlabel("Time [s]", fontsize=FONT_CONFIG['axis_label_size'])

format_scientific_axes(ax)

ax.legend(loc="upper right", fontsize=FONT_CONFIG['legend_size'])
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------------
# PLOT 2: NOISY INPUT DATA (Signal injection)
# -------------------------------------------------------------------------



TARGET_SNR_DB = 20.0
# Assuming 'add_noise_to_yout' is defined elsewhere
yout_noisy = add_noise_to_yout(yout, TARGET_SNR_DB)

fig, ax = plt.subplots(figsize=FIG_DIMENSIONS)

for i in range(3):
    ax.plot(tout, yout_noisy[:, i, 0], 
            label=f'{mass_labels[i]} (Noisy)', 
            color=colors[i], 
            linestyle='-',  
            alpha=0.7,  # Transparency added to visualize density of noise
            linewidth=plt.rcParams['lines.linewidth'] * 0.8) # Thinner lines for noisy data

ax.set_ylabel("Displacement [m]", fontsize=FONT_CONFIG['axis_label_size'])
ax.set_xlabel("Time [s]", fontsize=FONT_CONFIG['axis_label_size'])

format_scientific_axes(ax)

ax.legend(loc="upper right", fontsize=FONT_CONFIG['legend_size'])
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------------
# PLOT 3: MODEL PREDICTION (Denoised Output)
# -------------------------------------------------------------------------

# --- Data Reshaping (Tensor Reconstruction) ---
# Transforming stacked ML output back to State-Space Tensor: (Time, Outputs, Inputs)
n_samples_per_exp = len(t)
n_experiments = 3
n_outputs = 6

y_flat = Y_output_model.values
# 1. Reshape to separate experiments
y_reshaped_temp = y_flat.reshape((n_experiments, n_samples_per_exp, n_outputs))
# 2. Transpose to (Time, Output_Channels, Input_Excitation)
S_denoised = y_reshaped_temp.transpose(1, 2, 0)
# 3. Slice to isolate displacement variables
yout_denoised = S_denoised[:, 0:3, :]

fig, ax = plt.subplots(figsize=FIG_DIMENSIONS)

input_idx = 0  # Visualizing response to impulse on Input 1

for output_idx in range(3):
    ax.plot(tout, yout_denoised[:, output_idx, input_idx], 
            label=f'{mass_labels[output_idx]} (Denoised)', 
            color=colors[output_idx], 
            linestyle='-', 
            linewidth=plt.rcParams['lines.linewidth'])

ax.set_ylabel("Displacement [m]", fontsize=FONT_CONFIG['axis_label_size'])
ax.set_xlabel("Time [s]", fontsize=FONT_CONFIG['axis_label_size'])

format_scientific_axes(ax)

ax.legend(loc="upper right", fontsize=FONT_CONFIG['legend_size'])
plt.tight_layout()
plt.show()

In [ ]:
# --- Configurações Globais de Estilo Científico ---
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'], 
    'mathtext.fontset': 'stix',        
    'axes.linewidth': 1.2,             
    'xtick.direction': 'in',           
    'ytick.direction': 'in',           
    'xtick.major.size': 5,             
    'ytick.major.size': 5,
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,
    'xtick.top': True,                 
    'ytick.right': True,               
    'lines.linewidth': 2.0,            
    'legend.frameon': True,            
    'legend.framealpha': 0.9,          
    'legend.edgecolor': 'black',       
    'legend.fancybox': False,          
    'figure.dpi': 150                  
})

FONT_CONFIG = {
    'axis_label_size': 14, 
    'tick_label_size': 12, 
    'legend_size': 11,    
}

# Cores Profissionais (High Contrast)
colors = ['#0072B2', '#D55E00', '#009E73'] 

# Função auxiliar para formatar eixos
def format_scientific_axes(ax):
    ax.minorticks_on() 
    ax.grid(which='major', linestyle='--', linewidth=0.7, alpha=0.5, color='gray')
    ax.grid(which='minor', linestyle=':', linewidth=0.4, alpha=0.3, color='gray')
    ax.tick_params(axis='both', which='major', labelsize=FONT_CONFIG['tick_label_size'], pad=6)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)

# --- ALTERAÇÃO AQUI: Formato mais quadrado (Largura, Altura) ---
FIG_DIMENSIONS = (6, 5)

# -------------------------------------------------------------------------
# PLOT 1: CLEAN DATA (SOMENTE)
# -------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=FIG_DIMENSIONS)

mass_labels = ['Mass 1', 'Mass 2', 'Mass 3']

for i in range(3):
    ax.plot(tout, yout[:, i, 0], 
            label=f'{mass_labels[i]}', 
            color=colors[i],
            linestyle='-', 
            linewidth=plt.rcParams['lines.linewidth'])

# Configurações de Labels
ax.set_ylabel("Displacement [m]", fontsize=FONT_CONFIG['axis_label_size'], fontname='serif')
ax.set_xlabel("Time [s]", fontsize=FONT_CONFIG['axis_label_size'], fontname='serif')

# --- ALTERAÇÃO AQUI: Limite do eixo X em 8 segundos ---
ax.set_xlim(0, 8)

# Aplica formatação científica
format_scientific_axes(ax)

# Legenda posicionada
ax.legend(loc="upper right", fontsize=FONT_CONFIG['legend_size'])

plt.tight_layout()
plt.show()

In [ ]:
# --- Configurações Iniciais ---
lista_snrs = [30,25,20, 15, 12, 10, 5]
num_testes_por_snr = 10

# Definição das colunas (Baseado no seu shape de 6 canais: 3 posições + 3 acelerações)
# Considerando janela de atraso/avanço (p=2 -> 5 instantes no total)
input_cols = ['x1_m_2', 'x1_m_1', 'x1_p_1', 'x1_p_2', 
              'x2_m_2', 'x2_m_1', 'x2_p_1', 'x2_p_2', 
              'x3_m_2', 'x3_m_1', 'x3_p_1', 'x3_p_2', 
              'x4_m_2', 'x4_m_1', 'x4_p_1', 'x4_p_2', 
              'x5_m_2', 'x5_m_1', 'x5_p_1', 'x5_p_2', 
              'x6_m_2', 'x6_m_1', 'x6_p_1', 'x6_p_2']
output_cols = ['x1','x2','x3','x4','x5','x6']

config_model = {
    'latent_dim': 4, 
    'num_features_in': len(input_cols), 
    'num_features_out': len(output_cols)
}

results_data = []

# Supomos que 'S' (Dados Limpos Completos [10240, 6, 3]), 'yout' (Posição Limpa), 
# 't', 'mass_labels' e 'colors' já estejam definidos no ambiente.

# ==============================================================================
# LOOP PRINCIPAL (SNRs)
# ==============================================================================
for snr_val in lista_snrs:
    print(f"\n{'='*60}")
    print(f">>> Processando SNR Input: {snr_val} dB")
    print(f"{'='*60}")
    
    for teste_idx in range(1, num_testes_por_snr + 1):
        print(f"   -> Executando Teste {teste_idx}/{num_testes_por_snr}...")
        
        # 1. Gerar dados ruidosos (Aplica ruído em S: Posição + Aceleração)
        # Nota: Assume-se que a função add_noise_to_yout lida com o array S corretamente
        S_noisy = add_noise_to_yout(S, snr_val)
        
        # 2. Criar DataFrame Lagged
        dict_data = convert_yout_to_dict(S_noisy)
        lagged_df = create_lagged_dataframe(dict_data, num_points=2)
        
        # 3. Treinar Modelo (Reinicializa o modelo a cada iteração para treino do zero)
        model = _build_model_from_config(config_model)
        
        # verbose=0 para não poluir o output, ou ajuste conforme necessidade
        _, Y_output_model = train_final_model(model, lagged_df, input_cols, output_cols)
        
        # 4. Reshape do Output (Denoised)
        y_flat = Y_output_model.values
        n_samples_per_exp = len(t)
        n_experiments = 3
        n_outputs = 6 # (3 posições + 3 acelerações)
        
        # Reshape para (Experimentos, Amostras, Canais)
        y_reshaped_temp = y_flat.reshape((n_experiments, n_samples_per_exp, n_outputs))
        
        # Transpose para (Amostras, Canais, Experimentos) -> Formato S
        S_denoised = y_reshaped_temp.transpose(1, 2, 0)
        
        # Separa apenas a posição (canais 0, 1, 2) para cálculo do SNR de posição
        yout_denoised = S_denoised[:, 0:3, :]
        
        # 5. Cálculo SNR Output (Baseado apenas na Posição vs yout Limpo)
        # O "ruído" residual é a diferença entre a predição e o sinal perfeito
        residual_noise = yout_denoised - yout
        
        # Cálculo usando desvio padrão global
        snr_output_db = 20 * np.log10(np.std(yout) / np.std(residual_noise))
        delta_snr = snr_output_db - snr_val
        
        # Armazenar Resultados
        results_data.append({
            'SNR_Input': snr_val,
            'num_teste': teste_idx,
            'SNR_Output_denoising': snr_output_db,
            'Delta_SNR': delta_snr
        })

# ==============================================================================
# ANÁLISE ESTATÍSTICA E EXPORTAÇÃO
# ==============================================================================

df_resultados = pd.DataFrame(results_data)

print("\n" + "="*80)
print("ANÁLISE ESTATÍSTICA DOS RESULTADOS")
print("="*80)

# Lista para resumo estatístico
stats_summary = []

for snr in lista_snrs:
    # Filtrar dados para o SNR atual
    subset = df_resultados[df_resultados['SNR_Input'] == snr]['SNR_Output_denoising']
    
    media = subset.mean()
    desvio = subset.std()
    
    # Teste de Hipótese (T-Test para uma amostra)
    # H0: media_output <= SNR_Input
    # H1: media_output > SNR_Input (alternative='greater')
    t_stat, p_value = stats.ttest_1samp(subset, popmean=snr, alternative='greater')
    
    significativo = "SIM" if p_value < 0.05 else "NAO"
    
    stats_summary.append({
        'SNR_Input': snr,
        'Mean_Output': media,
        'Std_Dev': desvio,
        'Gain_Avg': media - snr,
        'P-Value': p_value,
        'Sig_Improvement': significativo
    })
    
    print(f"SNR Input {snr} dB: Média Output = {media:.2f} dB (+/- {desvio:.2f}) | Ganho Médio: {media - snr:.2f} dB")
    print(f"   -> Hipótese (Média > {snr})? {significativo} (p-value: {p_value:.5f})\n")

df_stats = pd.DataFrame(stats_summary)

# Exportação para Excel
filename = 'output_blind_denoising_low_SNR_aceleracao_posicao_10_testes_void_central.xlsx'

with pd.ExcelWriter(filename) as writer:  
    # Aba 1: Dados Brutos de cada teste
    df_resultados[['SNR_Input','num_teste','SNR_Output_denoising','Delta_SNR']].to_excel(writer, sheet_name='Raw_Data', index=False)
    
    # Aba 2: Resumo Estatístico
    df_stats.to_excel(writer, sheet_name='Statistics_Summary', index=False)

print(f"\nArquivo '{filename}' gerado com sucesso!")
print("\n=== Amostra dos Dados Brutos ===")
print(df_resultados.head(15))

In [ ]:
# --- Configurações de Estilo e Fonte (Personalizáveis) ---
FONT_CONFIG = {
    'axis_label_size': 14,
    'tick_label_size': 12,
    'legend_size': 12
}

# Nome do arquivo
file_path = 'output_blind_denoising_low_SNR_aceleracao_posicao_10_testes_void_central.xlsx'

try:
    # 1. Leitura do Arquivo Excel
    print(f"Lendo arquivo: {file_path}...")
    df = pd.read_excel(file_path, sheet_name='Raw_Data')

    # 2. Transformação dos Tipos de Dados
    df['SNR_Output_denoising'] = df['SNR_Output_denoising'].astype(float)
    df['Delta_SNR'] = df['Delta_SNR'].astype(float)
    df['SNR_Input'] = df['SNR_Input'].astype(int).astype(str)

    order_snr = ['5','10','12', '15', '20', '25', '30']

    # =========================================================================
    # GRÁFICO 1: Boxplot (Ganho de SNR por Classe de Input)
    # =========================================================================
    plt.figure(figsize=(10, 6))
    
    # Criação do Boxplot
    ax1 = sns.boxplot(x='SNR_Input', y='Delta_SNR', data=df, order=order_snr, palette="viridis")
    
    # Customização das Fontes e Eixos (SEM TÍTULO)
    plt.xlabel("Input SNR (dB)", fontsize=FONT_CONFIG['axis_label_size'])
    plt.ylabel("SNR Gain (dB)", fontsize=FONT_CONFIG['axis_label_size'])
    
    # Ajuste dos Ticks (Valores nos eixos)
    plt.xticks(fontsize=FONT_CONFIG['tick_label_size'])
    plt.yticks(fontsize=FONT_CONFIG['tick_label_size'])
    
    plt.grid(True, axis='y', alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.show()

    # =========================================================================
    # GRÁFICO 2: Histograma (Apenas para SNR Input = 20 dB)
    # =========================================================================
    
    # Filtrar dados para a classe '20'
    target_class = '20'
    df_20 = df[df['SNR_Input'] == target_class]
    
    if not df_20.empty:
        plt.figure(figsize=(10, 6))
        
        # Criação do Histograma
        sns.histplot(df_20['SNR_Output_denoising'], bins=4, kde=True, color='blue', edgecolor='black')
        
        # Customização das Fontes e Eixos (SEM TÍTULO)
        plt.xlabel("Output SNR (dB)", fontsize=FONT_CONFIG['axis_label_size'])
        plt.ylabel("Count / Frequency", fontsize=FONT_CONFIG['axis_label_size'])
        
        # Ajuste dos Ticks
        plt.xticks(fontsize=FONT_CONFIG['tick_label_size'])
        plt.yticks(fontsize=FONT_CONFIG['tick_label_size'])
        
        # Adiciona média visualmente (opcional)
        mean_val = df_20['SNR_Output_denoising'].mean()
        plt.axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.2f} dB')
        plt.legend(fontsize=FONT_CONFIG['legend_size'])
        
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Aviso: Não foram encontrados dados para SNR_Input = {target_class}")

except FileNotFoundError:
    print(f"Erro: O arquivo '{file_path}' não foi encontrado. Verifique o diretório.")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

In [ ]:
# --- Configurações Globais de Estilo Científico (Igual aos anteriores) ---
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'], 
    'mathtext.fontset': 'stix',        
    'axes.linewidth': 1.5,             
    'xtick.direction': 'in',           
    'ytick.direction': 'in',           
    'xtick.major.size': 6,             
    'ytick.major.size': 6,
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,
    'xtick.top': True,                 
    'ytick.right': True,               
    'lines.linewidth': 2.5,            
    'legend.frameon': True,            
    'legend.framealpha': 0.9,          
    'legend.edgecolor': 'black',       
    'legend.fancybox': False,          
    'figure.dpi': 150                  
})

FONT_CONFIG = {
    'axis_label_size': 18, # Mantendo padrão alto
    'tick_label_size': 14,
    'legend_size': 14
}

# Cores Profissionais (High Contrast)
# Substituindo: Blue -> C0, Red -> C1, Green -> C2, Purple -> C3
colors = {
    'blue': '#0072B2',   # Principal
    'red': '#D55E00',    # Destaque / Média
    'green': '#009E73',  # Referência
    'purple': '#CC79A7'  # Alternativo
}

# Função auxiliar para formatar eixos
def format_scientific_axes(ax):
    ax.minorticks_on() 
    # Grid: Major mais visível, Minor mais sutil
    ax.grid(which='major', linestyle='--', linewidth=0.7, alpha=0.5, color='gray')
    ax.grid(which='minor', linestyle=':', linewidth=0.4, alpha=0.3, color='gray')
    ax.tick_params(axis='both', which='major', labelsize=FONT_CONFIG['tick_label_size'], pad=6)
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)

# Nome do arquivo
file_path = 'output_blind_denoising_low_SNR_aceleracao_posicao_10_testes_void_central.xlsx'

try:
    # 1. Leitura do Arquivo Excel
    print(f"Lendo arquivo: {file_path}...")
    df = pd.read_excel(file_path, sheet_name='Raw_Data')

    # 2. Transformação dos Tipos de Dados
    df['SNR_Output_denoising'] = df['SNR_Output_denoising'].astype(float)
    df['Delta_SNR'] = df['Delta_SNR'].astype(float)
    df['SNR_Input'] = df['SNR_Input'].astype(int).astype(str)

    order_snr = ['5','10','12', '15', '20', '25']

    # =========================================================================
    # GRÁFICO 1.5: Boxplot Novo (Eixo Y partindo de 0)
    # =========================================================================
    print("Gerando Gráfico 1.5: Boxplot (Y partindo de 0)...")
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Boxplot
    sns.boxplot(x='SNR_Input', y='Delta_SNR', data=df, order=order_snr, 
                palette="viridis", ax=ax, linewidth=1.5)
    
    ax.set_xlabel("Input SNR (dB)", fontsize=FONT_CONFIG['axis_label_size'])
    ax.set_ylabel("SNR Gain (dB)", fontsize=FONT_CONFIG['axis_label_size'])
    
    # Força o eixo Y a começar em 0
    ax.set_ylim(bottom=-1, top=None) 

    format_scientific_axes(ax)
    
    plt.tight_layout()
    plt.show()

    # =========================================================================
    # GRÁFICO 2: Histograma (Apenas para SNR Input = 20 dB)
    # =========================================================================
    
    target_class = '20'
    df_20 = df[df['SNR_Input'] == target_class]
    
    if not df_20.empty:
        print(f"Gerando Gráfico 2: Histograma para SNR {target_class} dB...")
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Histograma com cor profissional
        sns.histplot(df_20['SNR_Output_denoising'], bins=4, kde=True, 
                     color=colors['blue'], edgecolor='black', ax=ax)
        
        # Configurações de Labels
        ax.set_xlabel("Output SNR (dB)", fontsize=FONT_CONFIG['axis_label_size'])
        ax.set_ylabel("Count / Frequency", fontsize=FONT_CONFIG['axis_label_size'])
        
        # 1. Definir início do eixo X
        ax.set_xlim(left=19.75)
        
        # 2. Reta vertical em 20dB (Input Reference)
        ax.axvline(x=20, color=colors['green'], linestyle='-', linewidth=2.5, 
                   label='SNR Input (20 dB)')
        
        # Linha da Média
        mean_val = df_20['SNR_Output_denoising'].mean()
        ax.axvline(mean_val, color=colors['red'], linestyle='--', linewidth=2.5,
                   label=f'Mean Output: {mean_val:.2f} dB')
        
        format_scientific_axes(ax)
        
        # Legenda (Center Upper)
        ax.legend(loc='upper center', fontsize=FONT_CONFIG['legend_size'])
        
        plt.tight_layout()
        plt.show()
    else:
        print(f"Aviso: Não foram encontrados dados para SNR_Input = {target_class}")

    # =========================================================================
    # GRÁFICO 3: Histograma Geral do Ganho (Delta_SNR) - Todos os Casos
    # =========================================================================
    print("Gerando Gráfico 3: Histograma Geral de Ganho (Delta_SNR)...")
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Histograma usando cor "Purple" da paleta profissional
    sns.histplot(df['Delta_SNR'], kde=True, 
                 color=colors['purple'], edgecolor='black', ax=ax)
    
    # Configurações de Labels
    ax.set_xlabel("Global SNR Gain (dB)", fontsize=FONT_CONFIG['axis_label_size'])
    ax.set_ylabel("Count / Frequency", fontsize=FONT_CONFIG['axis_label_size'])
    
    # Definir início do eixo X em -0.01
    ax.set_xlim(left=-0.01)
    
    # Adicionando linha de média geral
    global_mean = df['Delta_SNR'].mean()
    ax.axvline(global_mean, color=colors['red'], linestyle='--', linewidth=2.5,
               label=f'Global Mean Gain: {global_mean:.2f} dB')
    
    format_scientific_axes(ax)
    
    # Legenda (Left Upper conforme solicitado)
    ax.legend(loc='upper left', fontsize=FONT_CONFIG['legend_size'])
    
    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print(f"Erro: O arquivo '{file_path}' não foi encontrado. Verifique o diretório.")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

In [ ]:
# Supondo que o DataFrame 'df' já esteja carregado da etapa anterior.
# Caso precise carregar novamente:
# df = pd.read_excel('output_blind_denoising_low_SNR_aceleracao_posicao_10_testes_void_central.xlsx', sheet_name='Raw_Data')

# Garantir que SNR_Input e métricas numéricas estejam no formato correto
df['SNR_Input_Num'] = df['SNR_Input'].astype(float) # Coluna auxiliar numérica para ordenação/cálculo
df['SNR_Output_denoising'] = df['SNR_Output_denoising'].astype(float)
df['Delta_SNR'] = df['Delta_SNR'].astype(float)

stats_data = []

# Obter lista de SNRs únicos ordenados (do maior para o menor)
unique_snrs = sorted(df['SNR_Input_Num'].unique(), reverse=True)

for snr_target in unique_snrs:
    # Converter para string se necessário para filtrar o df original, ou usar a coluna numérica criada
    subset = df[df['SNR_Input_Num'] == snr_target]
    
    # Dados de interesse
    outputs = subset['SNR_Output_denoising']
    gains = subset['Delta_SNR']
    
    # 1. Estatísticas Descritivas
    mean_out = outputs.mean()
    std_out = outputs.std()
    
    mean_gain = gains.mean()
    median_gain = gains.median()
    max_gain = gains.max()
    min_gain = gains.min()
    
    # 2. Teste de Hipótese (T-Test)
    # H0: A média do Output é igual ao SNR de Entrada (não houve melhoria estatística)
    # H1: A média do Output é maior que o SNR de Entrada (houve melhoria)
    # alternative='greater' verifica se a amostra é maior que a média da população (snr_target)
    t_stat, p_val = stats.ttest_1samp(outputs, popmean=snr_target, alternative='greater')
    
    stats_data.append({
        'SNR Input (dB)': int(snr_target),
        'Média Output (dB)': mean_out,
        'Desvio Padrão (dB)': std_out,
        'Média Ganho (dB)': mean_gain,
        'Mediana Ganho (dB)': median_gain,
        'Mín Ganho (dB)': min_gain,
        'Máx Ganho (dB)': max_gain,
        'Estatística T': t_stat,
        'P-Valor': p_val
    })

# Criar DataFrame Final
df_stats_detalhada = pd.DataFrame(stats_data)

# Configurar índice
df_stats_detalhada.set_index('SNR Input (dB)', inplace=True)

# Exibir a tabela formatada
print("=== Tabela Estatística Detalhada por Nível de Ruído ===")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Função de estilo para destacar p-valores significativos (< 0.05) e formatar floats
def highlight_p_val(s):
    is_significant = s < 0.05
    return ['font-weight: bold; color: green' if v else '' for v in is_significant]

# Exibição simples no console
print(df_stats_detalhada.round(4))

# Se estiver no Jupyter/Colab, a linha abaixo gera uma visualização HTML bonita:
# display(df_stats_detalhada.style.format("{:.4f}").apply(highlight_p_val, subset=['P-Valor']))

In [ ]:
filename = 'output_blind_denoising_low_SNR_aceleracao_posicao_10_testes_void_central.xlsx'

# Verifica se o arquivo existe antes de tentar abrir em modo append
if os.path.exists(filename):
    # mode='a' (append) adiciona uma nova aba
    # if_sheet_exists='replace' substitui a aba 'statistics' se ela já existir
    with pd.ExcelWriter(filename, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df_stats_detalhada.to_excel(writer, sheet_name='statistics')
    print(f"Sucesso: Tabela salva na aba 'statistics' do arquivo existente '{filename}'.")
else:
    # Se o arquivo não existir por algum motivo, cria um novo
    with pd.ExcelWriter(filename, engine='openpyxl', mode='w') as writer:
        df_stats_detalhada.to_excel(writer, sheet_name='statistics')
    print(f"Aviso: Arquivo não encontrado. Novo arquivo '{filename}' criado com a aba 'statistics'.")

# (5) Study parameter identification by classical methods as the signal-to-noise ratio decreases, comparing noisy signals and machine learning model outputs.

In [ ]:
import pandas as pd
import numpy as np

# --- 1. Fixed Configurations & Setup ---

snr_levels_list = [20, 15]

# Feature Engineering: Lagged variables for time-series context
input_cols = [
    'x1_m_2', 'x1_m_1', 'x1_p_1', 'x1_p_2',
    'x2_m_2', 'x2_m_1', 'x2_p_1', 'x2_p_2',
    'x3_m_2', 'x3_m_1', 'x3_p_1', 'x3_p_2',
    'x4_m_2', 'x4_m_1', 'x4_p_1', 'x4_p_2',
    'x5_m_2', 'x5_m_1', 'x5_p_1', 'x5_p_2',
    'x6_m_2', 'x6_m_1', 'x6_p_1', 'x6_p_2'
]

config_model = {
    'latent_dim': 4, 
    'num_features_in': len(input_cols), 
    'num_features_out': len(output_cols)
}

results_data_list = []

# --- 2. Helper Function (Defined OUTSIDE the loop) ---
def execute_modal_identification_and_log(
    signal_tensor: np.ndarray, 
    condition_label: str,
    current_snr_input: float,
    current_snr_output: float,
    snr_gain: float
):
    """
    Performs modal identification (LSCEM and ITD), matches estimated modes 
    to ground truth, and appends results to the global list.
    """
    

    # --- Method A: Least Squares Complex Exponential (LSCEM) ---
    decimation_factor = 20
    fs_id = fs / decimation_factor
    
    # Extract Input 1 (Driving Point), All Channels, Decimated
    y_lscem = signal_tensor[::decimation_factor, :, 0] 
    
    poles_lscem = least_squares_complex_exponential_method(y_lscem, fs_id, order=6)
    params_lscem = get_modal_params(poles_lscem)
    params_lscem.sort(key=lambda x: x['wn']) # Sort by frequency
    
    # --- Method B: Ibrahim Time Domain (ITD) ---
    # Extract Input 1, All Channels, Full Sampling Rate
    y_itd = signal_tensor[:, :, 0] 
    delay_s = 20
    
    poles_itd, modes_itd = ibrahim_time_domain(y_itd, fs, delay_block=delay_s)
    params_itd = process_modal_results(poles_itd, modes_itd, fs)
    
    # ITD Uniqueness Filter (Remove duplicate/spurious modes close to each other)
    params_itd_unique = []
    last_freq = -1
    for p in params_itd:
        if abs(p['wn'] - last_freq) > 0.2:
            params_itd_unique.append(p)
            last_freq = p['wn']
    params_itd_unique.sort(key=lambda x: x['wn'])

    # --- Compare with Ground Truth & Log Results ---
    methods_to_evaluate = [
        ('LSCEM', params_lscem), 
        ('ITD', params_itd_unique)
    ]
    
    for method_name, estimated_params in methods_to_evaluate:
        # Iterate over known Ground Truth modes (Real Wn and Zeta)
        for i, (wn_real, zeta_real) in enumerate(zip(lista_wn_real, lista_zeta_real)):
            wn_est, zeta_est = np.nan, np.nan
            
            # Attempt to match the i-th ordered mode
            if i < len(estimated_params):
                candidate = estimated_params[i]
                
                # Safety check: Is the candidate frequency within a reasonable window?
                # (Prevents matching a spurious 50Hz noise mode to a 10Hz physical mode)
                if abs(candidate['wn'] - wn_real) < 10.0:
                    wn_est = candidate['wn']
                    zeta_est = candidate['zeta']
            
            # Calculate Percentage Errors
            if not np.nan:
                err_wn = 100 * abs(wn_est - wn_real) / wn_real
                err_zeta = 100 * abs(zeta_est - zeta_real) / zeta_real
            else:
                err_wn, err_zeta = np.nan, np.nan
            
            # Append entry to global results
            results_data_list.append({
                'SNR_Input_dB': current_snr_input,
                'SNR_Output_dB': current_snr_output,
                'SNR_Gain_dB': snr_gain,
                'Condition': condition_label,  # 'Noisy' or 'Denoised'
                'Method': method_name,         # 'LSCEM' or 'ITD'
                'Mode_Order': i + 1,
                'Wn_Real_Hz': wn_real,
                'Wn_Est_Hz': wn_est,
                'Error_Wn_%': err_wn,
                'Zeta_Real': zeta_real,
                'Zeta_Est': zeta_est,
                'Error_Zeta_%': err_zeta
            })

# --- 3. Main Processing Loop ---

print(f"--- Starting Sensitivity Analysis across {len(snr_levels_list)} Noise Levels ---")

for snr_val in snr_levels_list:
    print(f"\n>>> Processing Input SNR: {snr_val} dB")
    
    # A. Generate Noisy Data (Simulation)
    # S is assumed to be the global clean source tensor
    S_noisy_tensor = add_noise_to_yout(S, snr_val) 
    
    # B. Pre-processing for Machine Learning (Lag Generation)
    dict_data = convert_yout_to_dict(S_noisy_tensor)
    lagged_df = create_lagged_dataframe(dict_data, num_points=2)
    
    # C. Model Training & Inference
    # Re-initialize model to prevent weight leakage between SNR levels
    model = _build_model_from_config(config_model)
    _, Y_output_model = train_final_model(model, lagged_df, input_cols, output_cols)
    
    # D. Tensor Reshaping (Flattened ML Output -> State Space Tensor)
    # Shape: (Total Samples, Features) -> (Experiments, Samples, Channels)
    y_flat = Y_output_model.values
    n_samples_per_exp = len(t)
    
    # 3 Experiments (Inputs), n_samples, 6 Output Channels
    y_reshaped_temp = y_flat.reshape((3, n_samples_per_exp, 6))

    # Transpose to standard Control format: (Time, Channels, Experiments)
    S_denoised = y_reshaped_temp.transpose(1, 2, 0)
    
    # Slice to keep only displacement variables if necessary (First 3 channels)
    yout_denoised = S_denoised[:, 0:3, :]
    yout_noisy_slice = S_noisy_tensor[:, 0:3, :] # Slice noisy data to match denoised shape for fairness
    
    # E. Calculate Denoising Performance (Output SNR)
    residual_noise = yout_denoised - yout # 'yout' is the global ground truth
    
    # SNR = 20 * log10(Signal_RMS / Noise_RMS)
    snr_output_db = 20 * np.log10(np.std(yout) / np.std(residual_noise))
    delta_snr = snr_output_db - snr_val
    
    print(f"    Denoising Result: Input {snr_val:.2f} dB -> Output {snr_output_db:.2f} dB (Gain: +{delta_snr:.2f} dB)")
    
    # F. Run Modal Identification on BOTH Datasets
    # 1. Evaluate Noisy Data (Baseline)
    execute_modal_identification_and_log(
        signal_tensor=yout_noisy_slice, 
        condition_label='Noisy Input',
        current_snr_input=snr_val,
        current_snr_output=snr_val, # For noisy data, output SNR is conceptually same as input
        snr_gain=0.0
    )
    
    # 2. Evaluate Denoised Data (Model Result)
    execute_modal_identification_and_log(
        signal_tensor=yout_denoised, 
        condition_label='Denoised Output',
        current_snr_input=snr_val,
        current_snr_output=snr_output_db,
        snr_gain=delta_snr
    )

# --- 4. Final Data Aggregation & Display ---

df_general_results = pd.DataFrame(results_data_list)

print("\n=== Consolidated Results Summary (Sample) ===")
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

columns_to_show = [
    'SNR_Input_dB', 'Condition', 'Method', 'Mode_Order', 
    'Wn_Real_rad_s', 'Wn_Est_rad_s', 'Error_Wn_%', 
    'Zeta_Real', 'Zeta_Est', 'Error_Zeta_%'
]

# Displaying a subset to verify
print(df_general_results[columns_to_show])

In [ ]:
import openpyxl
df2 = df_general_results.copy()
with pd.ExcelWriter('output_blind_denoising_modal_parameters_low_SNR_aceleracao_posicao_void_central.xlsx') as writer:  
    df2.to_excel(writer, sheet_name='Sheet_name_1')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyBboxPatch, Rectangle, Polygon, Circle
import matplotlib.patches as mpatches

# --- Configurações de Tamanho de Fonte (Aumentadas) ---
fs_title = 24      
fs_header = 18     
fs_label = 16      
fs_math = 15       
fs_big_math = 22   
fs_note = 12       
fs_process = 16    

# Configuração da figura
# Ajustei a altura de 8 para 7 e o ylim para cortar a parte inferior vazia
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(2.8, 10.5) # Corte em 2.8 para remover margem inferior
ax.axis('off')

# Cores
input_color = '#3498db'     # Azul para input
encoder_color = '#2ecc71'   # Verde para encoder
latent_color = '#e74c3c'    # Vermelho para latent space
decoder_color = '#9b59b6'   # Roxo para decoder
output_color = '#f39c12'    # Laranja para output
arrow_color = '#2c3e50'     # Cinza escuro para setas

# Título
ax.text(7, 9.8, 'Time-Window Noise2Noise', 
        fontsize=fs_title, fontweight='bold', ha='center', color='#2c3e50')

# ========== NOISY INPUT WINDOW ==========
ax.text(2.5, 8.4, 'Noisy Input Window', fontsize=fs_header, fontweight='bold', 
        ha='center', color=input_color)

# Retângulo do input window
input_box = FancyBboxPatch((0.8, 5.5), 3.4, 2.3, boxstyle="round,pad=0.1",
                            facecolor='white', edgecolor=input_color, linewidth=3)
ax.add_patch(input_box)

# Vetores de entrada
time_labels = [r'$X(t_{i-p})$', r'$\vdots$', r'$X(t_{i-1})$'
, r'$X(t_{i+1})$', r'$\vdots$', r'$X(t_{i+p})$']

y_positions = np.linspace(7.2, 6.0, 6) # Espalhei um pouco mais verticalmente
for i, (label, y) in enumerate(zip(time_labels, y_positions)):
    x_pos = 2.5
    
    # Pedido 3: Todos na mesma tonalidade de azul (input_color)
    # Removida a lógica que deixava o termo central vermelho
    if i == 3:
        # Mantive bold apenas para indicar que é o vetor central, mas na cor azul
        ax.text(x_pos, y, label, fontsize=fs_math + 1, ha='center', fontweight='bold', color=input_color)
    else:
        ax.text(x_pos, y, label, fontsize=fs_math, ha='center', color=input_color)

# Adicionar chaves para mostrar o vetor
ax.plot([1.8, 1.8], [7.4, 5.8], color=input_color, linewidth=1.5)
ax.plot([3.2, 3.2], [7.4, 5.8], color=input_color, linewidth=1.5)

ax.text(2.5, 5.1, r'$\mathbf{X}_{i,p}$', fontsize=fs_label, ha='center', 
        color=input_color, fontweight='bold')
ax.text(2.5, 4.8, f'(Window size: 2p vectors)', fontsize=fs_note, ha='center', 
        color=input_color, style='italic')

# ========== ENCODER FUNNEL (HORIZONTAL) ==========
ax.text(5.8, 8.4, 'Encoder', fontsize=fs_header, fontweight='bold', 
        ha='center', color=encoder_color)

# Coordenadas para funil horizontal (esquerda -> direita)
enc_x_start = 4.8
enc_x_end = 6.8
enc_h_start = 2.0  # Altura na entrada
enc_h_end = 0.6    # Altura na saída (gargalo)
y_center = 6.6

encoder_funnel = Polygon([
    (enc_x_start, y_center + enc_h_start/2),  # Topo esquerdo
    (enc_x_start, y_center - enc_h_start/2),  # Base esquerda
    (enc_x_end, y_center - enc_h_end/2),      # Base direita
    (enc_x_end, y_center + enc_h_end/2),      # Topo direito
], closed=True, facecolor=encoder_color, alpha=0.2, edgecolor=encoder_color, linewidth=2)

ax.add_patch(encoder_funnel)

# Linhas internas do funil
num_lines = 5
for i in range(num_lines):
    ratio = i / (num_lines - 1)
    x = enc_x_start + ratio * (enc_x_end - enc_x_start)
    h = enc_h_start - ratio * (enc_h_start - enc_h_end)
    
    if i < num_lines - 1: # Linha vertical
        ax.plot([x, x], [y_center - h/2, y_center + h/2], 
                color=encoder_color, alpha=0.5, linewidth=0.8)

# Setas de compressão
for i in range(3):
    y_pos = y_center + (i-1)*0.3 
    ax.annotate('', xy=(enc_x_end - 0.2, y_pos), xytext=(enc_x_start + 0.2, y_pos),
                arrowprops=dict(arrowstyle='->', color=encoder_color, 
                              lw=1.5, alpha=0.7))

# ========== LATENT SPACE ==========
ax.text(7.5, 8.4, 'Latent Space', fontsize=fs_header, fontweight='bold', 
        ha='center', color=latent_color)

# Círculo
latent_radius = 0.45 # Aumentei levemente o raio
latent_circle = Circle((7.5, y_center), latent_radius, facecolor=latent_color, 
                       alpha=0.3, edgecolor=latent_color, linewidth=3)
ax.add_patch(latent_circle)

# Texto dentro do círculo
ax.text(7.5, y_center, 'r', fontsize=fs_big_math, fontweight='bold', ha='center', 
        va='center', color=latent_color)

ax.text(7.5, 5.3, 'Bottleneck', fontsize=fs_process, ha='center', 
        color=latent_color, fontweight='bold')

# ========== DECODER FUNNEL (HORIZONTAL) ==========
ax.text(9.2, 8.4, 'Decoder', fontsize=fs_header, fontweight='bold', 
        ha='center', color=decoder_color)

# Coordenadas
dec_x_start = 8.2
dec_x_end = 10.2
dec_h_start = 0.6  
dec_h_end = 2.0    
y_center = 6.6

decoder_funnel = Polygon([
    (dec_x_start, y_center + dec_h_start/2),  
    (dec_x_start, y_center - dec_h_start/2),  
    (dec_x_end, y_center - dec_h_end/2),      
    (dec_x_end, y_center + dec_h_end/2),      
], closed=True, facecolor=decoder_color, alpha=0.2, edgecolor=decoder_color, linewidth=2)

ax.add_patch(decoder_funnel)

# Linhas internas
for i in range(num_lines):
    ratio = i / (num_lines - 1)
    x = dec_x_start + ratio * (dec_x_end - dec_x_start)
    h = dec_h_start + ratio * (dec_h_end - dec_h_start)
    
    if i > 0: 
        ax.plot([x, x], [y_center - h/2, y_center + h/2], 
                color=decoder_color, alpha=0.5, linewidth=0.8)

# Setas de expansão
for i in range(3):
    y_pos = y_center + (i-1)*0.3
    ax.annotate('', xy=(dec_x_end - 0.2, y_pos), xytext=(dec_x_start + 0.2, y_pos),
                arrowprops=dict(arrowstyle='->', color=decoder_color, 
                              lw=1.5, alpha=0.7))

# ========== TARGET OUTPUT ==========
ax.text(12.5, 8.4, 'Target Output', fontsize=fs_header, fontweight='bold', 
        ha='center', color=output_color)

# Retângulo do output
output_box = FancyBboxPatch((11.8, 6.1), 1.5, 1.1, boxstyle="round,pad=0.1",
                            facecolor='white', edgecolor=output_color, linewidth=3)
ax.add_patch(output_box)

# Vetor de saída
ax.text(12.5, 6.7, r'$\mathbf{X(t_i)}$', fontsize=fs_big_math, ha='center', 
        fontweight='bold', color=output_color)
ax.text(12.5, 6.35, 'Clean signal', fontsize=fs_note+2, ha='center', 
        color=output_color, style='italic')

# ========== SETAS DE CONEXÃO ==========
# Ajustadas para a nova altura y_center = 6.6
arrow_y = 6.6
ax.annotate('', xy=(4.8, arrow_y), xytext=(4.2, arrow_y),
            arrowprops=dict(arrowstyle='->', color=arrow_color, lw=2.5))

ax.annotate('', xy=(7.05, arrow_y), xytext=(6.8, arrow_y),
            arrowprops=dict(arrowstyle='->', color=arrow_color, lw=2.5))

ax.annotate('', xy=(8.2, arrow_y), xytext=(7.95, arrow_y),
            arrowprops=dict(arrowstyle='->', color=arrow_color, lw=2.5))

ax.annotate('', xy=(11.8, arrow_y), xytext=(10.2, arrow_y),
            arrowprops=dict(arrowstyle='->', color=arrow_color, lw=2.5))

# ========== FLUXO DE PROCESSO SIMPLIFICADO ==========
process_y = 3.5
process_labels = ['Temporal\nContext', 'Compressed\nRepresentation', 'Denoised\nOutput']
process_colors = [input_color, latent_color, output_color]
process_x_positions = [2.5, 7.5, 12.5] 


for i, (x, label, color) in enumerate(zip(process_x_positions, process_labels, process_colors)):
    # Texto do processo (Fonte aumentada)
    ax.text(x, process_y, label, fontsize=fs_process, ha='center', fontweight='bold', 
            color=color, bbox=dict(boxstyle="round,pad=0.3", facecolor='white', 
                                   edgecolor=color, alpha=0.9))
    
    if i < len(process_x_positions) - 1:
        next_x = process_x_positions[i+1]
        ax.annotate('', xy=(next_x - 1.2, process_y), xytext=(x + 1.2, process_y),
                   arrowprops=dict(arrowstyle='->', color=arrow_color, lw=1.5))

plt.tight_layout()
# Salvar
plt.savefig('autoencoder_window_schema_final_void.png', dpi=300, bbox_inches='tight', transparent=False)
plt.show()